> Experiments with [plot](../trapallada/plot.py) module

In [1]:
import pathlib
import tempfile

import pandas as pd

import jax.random as jr
import jax.numpy as jnp

import trapallada.data
import trapallada.plot

So that imported modules are automatically reloaded

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Data directory

In [3]:
data_directory = pathlib.Path.cwd().parent / 'samples' / 'data' / 'mini_rate'
display(data_directory)
assert data_directory.exists()

PosixPath('/home/manu/Sync/git/trapallada/samples/data/mini_rate')

Individual files with

- the actual signals, and

- a `DataFrame` with metadata.

In [4]:
signals_file = data_directory / f'signals.pt'
assert signals_file.exists()

masks_file = data_directory / 'masks.pt'
assert masks_file.exists()

metadata_file = data_directory / f'metadata.parquet'
assert metadata_file.exists

A *PyTorch* `Dataset`

In [5]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, masks_file)
len(ds)

148

# `signals_array`

A few signals are extracted from the above `Dataset`

In [6]:
ds.signals.shape

torch.Size([148, 1, 225])

In [7]:
sample_signals = ds.signals[:3, 0, ]
sample_signals.shape

torch.Size([3, 225])

Only the first `1,000` signals are kept

In [8]:
n_samples = 1_000
sample_signals = sample_signals[:, :n_samples]
trapallada.plot.signals_array(sample_signals)

In [9]:
trapallada.plot.signals_array(sample_signals, columns=['Foo', 'Faa', 'Fii'])

# `df`

Above signals are turned into a `DataFrame`...

In [10]:
df = pd.DataFrame(sample_signals.T, columns=[f'signal_{i}' for i in range(len(sample_signals)) ])
df.head()

,signal_0,signal_1,signal_2
0,0.249934,0.989075,-0.749298
1,0.536234,-0.953315,0.242648
2,0.179131,-1.269671,-0.150724
3,2.090136,-1.316452,0.332629
4,-1.769218,-1.247504,1.358906


...for plotting

In [11]:
trapallada.plot.df(df)

Passing in custom names for the columns

In [12]:
trapallada.plot.df(df, columns=['A', 'B', 'C'])

# pairwise_signals_comparison

In [13]:
x, x_pred = [jr.normal(key=jr.PRNGKey(seed), shape=(5, 10)) for seed in [1, 2]]
trapallada.plot.pairwise_signals_comparison(
    jnp.arange(x.shape[1]),
    x,
    x_pred,
    trapallada.xsa.util.rates_columns, [f'Reconstr. {e}' for e in trapallada.xsa.util.rates_columns],
    'Actual vs reconstruction', 't', 'x')

# full_signals

In [14]:
sample_obs = '0404720301'
# sample_obs = '0400340101'
with tempfile.TemporaryDirectory(delete=False) as tmp_dir:

    # context's descriptor is turned into a `Pathlib`
    tmp_dir_root = pathlib.Path(tmp_dir)

    tar_file = trapallada.xsa.access.download_obs(sample_obs, output_directory=tmp_dir_root)

INFO: https://nxsa.esac.esa.int/nxsa-sl/servlet/data-action-aio?obsno=0404720301&level=PPS&instname=PN [astroquery.esa.xmm_newton.core]
INFO: Wrote https://nxsa.esac.esa.int/nxsa-sl/servlet/data-action-aio?obsno=0404720301&level=PPS&instname=PN to /tmp/tmpk5eavhae/0404720301.tar [astroquery.esa.xmm_newton.core]


In [15]:
arguments = [tmp_dir_root.as_posix()]
with tempfile.TemporaryDirectory(delete=False) as tmp_output_dir:
    trapallada.xsa.util.cl_tars_to_array(arguments + [tmp_output_dir, 'sample'] + ['--signals-to-extract'] + trapallada.xsa.util.rates_columns + trapallada.xsa.util.background_columns)
tmp_output_dir

trapallada.xsa.util - INFO - Requested a window size of, effectively, duration 18396.0 seconds (period is 91.98 seconds, step is 4599.0 seconds)
trapallada.xsa.util - INFO - 20 signals of length 200 samples (18304.02 seconds)


'/tmp/tmpnwsunf4b'

In [16]:
signals_df, mask_series = trapallada.xsa.util.get_full_signals('0404720301/P0404720301PNX000SRCTSR8001', pathlib.Path(tmp_output_dir) / 'sample' / 'full_signals')
signals_df.head()

data,RATE1,RATE2,RATE3,RATE4,RATE5,BACK1V,BACK2V,BACK3V,BACK4V,BACK5V
TIME,,,,,,,,,,
2007-01-15 18:57:23.569320,-0.294014,-0.566235,2.378593,1.883989,4.425532,5.170950,4.194507,5.047898,3.653525,5.075870
2007-01-15 18:58:55.549320,-0.294014,-0.566235,-0.770621,3.502937,3.082441,1.781433,4.575751,5.056063,1.927038,2.880096
2007-01-15 19:00:27.529320,-0.294014,6.079143,2.344473,2.161434,8.116025,3.620308,5.617491,2.413972,4.404505,5.206001
2007-01-15 19:01:59.509320,7.610776,2.797909,-0.770621,-0.681765,4.679399,-0.068100,2.884643,3.036111,1.819009,2.805902
2007-01-15 19:03:31.489320,-0.294014,-0.566235,0.438256,2.052385,6.679115,4.054274,3.865352,2.409992,2.728865,4.914553


In [17]:
trapallada.plot.full_signals('0404720301/P0404720301PNX000SRCTSR8001', pathlib.Path(tmp_output_dir) / 'sample' / 'full_signals', *signals_df.index[[100, 150]])